In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
import torch.nn as nn
from pyro.infer import Predictive

###PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

###Forward and Backward Selection
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
import statsmodels.api as sm

# HMC
from pyro.infer import MCMC, NUTS

# variational inference
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from tqdm.auto import trange
from tqdm.notebook import trange

import matplotlib as mpl
import os
import sys
import math

C:\Users\thumo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Setting Direction for Optbnn file
sys.path.append(
    r"C:\Users\thumo\OneDrive - Georgia Institute of Technology\Georgia Tech\Semesters\Spring 2025\CSE 8803 IUQ\Project\2-BNN_trained_prior\you-need-a-good-prior"
)

In [3]:
from optbnn.gp.models.gpr import GPR
from optbnn.gp import kernels, mean_functions, priors
from optbnn.bnn.reparam_nets import GaussianMLPReparameterization
from optbnn.bnn.nets.mlp import MLP
from optbnn.bnn.likelihoods import LikGaussian
from optbnn.bnn.priors import FixedGaussianPrior, OptimGaussianPrior
from optbnn.prior_mappers.wasserstein_mapper import MapperWasserstein, WassersteinDistance
from optbnn.utils.rand_generators import MeasureSetGenerator, GridGenerator
from optbnn.utils.normalization import normalize_data
from optbnn.utils.exp_utils import get_input_range
from optbnn.metrics.sampling import compute_rhat_regression
from optbnn.metrics import uncertainty as uncertainty_metrics
from optbnn.sgmcmc_bayes_net.regression_net import RegressionNet
from optbnn.utils import util

In [4]:
###IGNORE THIS IF YOU DON'T HAVE CUDA
import torch

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("CUDA Device Name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
print(torch.version.cuda)
print(torch.cuda.device_count())

CUDA Available: False
CUDA Device Count: 0
Training on device: cpu
None
0


In [5]:
class BNN(PyroModule):
    def __init__(self, weight_prior, bias_prior, in_dim=1, out_dim=1, hid_dim=10, n_hid_layers=5):
        '''
        functional model (network architecture):
            a fully connected neural network.

        stochastic model:
            Gaussian prior on weight and bias: p(theta) ~ dist.Normal(0., weight_prior or bias_prior), where weight_prior and bias_prior are learned;
            Gaussian likelihood function: p(y | x, theta) ~ dist.Normal(functional model(x), sigma^2), where sigma ~ dist.Gamma(.5, 1).
        '''
        super().__init__()

        # make sure the dimensions are valid
        assert in_dim > 0 and out_dim > 0 and hid_dim > 0 and n_hid_layers > 0

        # activation function for the whole network, can also be ReLU or LeakyReLU
        self.activation = nn.Tanh()

        # define the layer sizes and the PyroModule layer list
        self.layer_sizes = [in_dim] + n_hid_layers * [hid_dim] + [out_dim]
        layer_list = [PyroModule[nn.Linear](self.layer_sizes[idx - 1], self.layer_sizes[idx]) for idx in
                      range(1, len(self.layer_sizes))]
        self.layers = PyroModule[torch.nn.ModuleList](layer_list)

        # set the probability distribution for each layer's weight and bias
        for layer_idx, layer in enumerate(self.layers):
            layer.weight = PyroSample(dist.Normal(0., weight_prior[layer_idx]).expand([self.layer_sizes[layer_idx + 1], self.layer_sizes[layer_idx]]).to_event(2))
            layer.bias = PyroSample(dist.Normal(0., bias_prior[layer_idx]).expand([self.layer_sizes[layer_idx + 1]]).to_event(1))

    def forward(self, x, y=None):
        # functional model(x)
        # input --> hidden
        x = self.activation(self.layers[0](x))
        # hidden --> hidden
        for layer in self.layers[1:-1]:
            x = self.activation(layer(x))
        # hidden --> output
        mu = self.layers[-1](x).squeeze()

        # sample from P(y | x, \theta)
        sigma = pyro.sample("sigma", dist.Gamma(.5, 1))
        with pyro.plate("data", x.shape[0]):
            # obs is used when quantifying and visualizing the uncertainty of predictions
            obs = pyro.sample("obs", dist.Normal(mu, sigma * sigma), obs=y)
        
        return mu

In [6]:
def plot_predictions(preds, y):
    '''
    Function to visualize the predictions and the uncertainty of predictions.
    '''
    y_pred = preds['obs'].T.detach().numpy().mean(axis=1)
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))

    # decide the range of the y axis based on the number of the labels
    time_idx = np.array(range(len(y)))
    xlims = [time_idx.min() - 0.1, time_idx.max() + 0.1]
    # decide the range of the y axis based on the range of the labels
    ylims = [min(y.min(), y_pred.min()) - 20,
             max(y.max(), y_pred.max()) + 20]
    
    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("time", fontsize=20)
    plt.ylabel("closing price", fontsize=20)

    ax.plot(time_idx, y, 'ko', markersize=1, label="observations")
    ax.plot(time_idx, y_pred, '-', linewidth=0.5, color="#408765", label="predictive mean")
    ax.fill_between(time_idx, y_pred - 2 * y_std, y_pred + 2 * y_std, alpha=0.6, color='#86cfac', zorder=0)

    plt.legend(loc=4, fontsize=15, frameon=False)

In [7]:
def plot_uncertainty(preds, y):
    '''
    Function to visualize only the uncertainty.
    '''
    fig, ax = plt.subplots(figsize=(10, 5))

    time_idx = np.array(range(len(y)))
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    xlims = [time_idx.min() - 0.1, time_idx.max() + 0.1]
    ylims = [y_std.min() - 0.5, y_std.max() + 0.5]

    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("time", fontsize=20)
    plt.ylabel("std of closing price", fontsize=20)

    ax.plot(time_idx, y_std, 'ko', markersize=1, label="std of predictions")
    ax.plot(time_idx, y_std, '-', linewidth=0.5, color="#408765")

    plt.legend(loc=4, fontsize=15, frameon=False)

### BNN Prior: Ericcson

In [8]:
df_pre_ericcson = pd.read_csv("pregdprApril2016_Ericcson.csv")
df_pre_ericcson.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,target
0,261755983.0,101.0,10.0,1.0,1.535461,261755983.0,102.8,10.0,1.0,1.535461,...,103.8,10.0,1.0,1.535461,261755983.0,104.8,10.0,1.0,1.535461,103.603225
1,261755983.0,102.8,10.0,1.0,1.535461,261755983.0,103.0,10.0,1.0,1.535461,...,104.8,10.0,1.0,1.535461,261755983.0,106.1,10.0,1.0,1.535461,104.420383
2,261755983.0,103.0,10.0,1.0,1.535461,261755983.0,103.8,10.0,1.0,1.535461,...,106.1,10.0,1.0,1.535461,261755983.0,106.9,10.0,1.0,1.535461,100.434081
3,261755983.0,103.8,10.0,1.0,1.535461,261755983.0,104.8,10.0,1.0,1.535461,...,106.9,10.0,1.0,1.535461,261755983.0,103.5,10.0,1.0,1.585659,99.333622
4,261755983.0,104.8,10.0,1.0,1.535461,261755983.0,106.1,10.0,1.0,1.535461,...,103.5,10.0,1.0,1.585659,261755983.0,102.2,10.0,1.0,1.585659,98.457528


In [9]:
##Extract Values
X_pre_ericcson = df_pre_ericcson.iloc[:, :-1].values
y_pre_ericcson = df_pre_ericcson["target"].values

# Create a mask for rows without NaNs in either X or y
valid_mask = ~np.isnan(X_pre_ericcson).any(axis=1) & ~np.isnan(y_pre_ericcson)

# Apply mask to both X and y
X_pre_ericcson = X_pre_ericcson[valid_mask]
y_pre_ericcson = y_pre_ericcson[valid_mask]
X_pre_ericcson, X_pre_ericcson.shape, type(X_pre_ericcson)


(array([[2.61755983e+08, 1.01000000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.53546105e+00],
        [2.61755983e+08, 1.02800000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.53546105e+00],
        [2.61755983e+08, 1.03000000e+02, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.53546105e+00],
        ...,
        [2.61755983e+08, 7.53500000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.66165543e+00],
        [2.61755983e+08, 7.60000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.66165543e+00],
        [2.61755983e+08, 6.80000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.66165543e+00]], shape=(264, 25)),
 (264, 25),
 numpy.ndarray)

In [10]:
y_pre_ericcson, y_pre_ericcson.shape, type(y_pre_ericcson)

(array([103.6032248 , 104.42038302, 100.43408099,  99.33362165,
         98.45752819, 100.0186524 , 101.23125207, 101.10614873,
         93.03767666,  92.01955163,  90.96999347,  89.93608061,
         90.25991603,  87.95710513,  87.41993472,  87.75270075,
         86.85316497,  87.9168448 ,  89.76066882,  88.83371273,
         88.29504344,  90.03784912,  90.23791613,  89.99534341,
         88.76077095,  88.62378663,  87.26629422,  86.80584723,
         88.46523094,  87.75840681,  87.89424829,  90.5017429 ,
         89.80511913,  89.71096092,  90.70142546,  89.01046448,
         91.48295834,  91.87129964,  90.25780765,  90.43761205,
         89.6759953 ,  88.69277064,  89.58862255,  87.9351845 ,
         89.01214079,  88.56280145,  87.19603858,  86.23256538,
         87.62017195,  86.0457368 ,  86.28628625,  85.95795004,
         85.29986772,  85.01117751,  86.09688731,  85.43994313,
         85.18051445,  85.62657   ,  85.69985457,  85.65977043,
         85.76486981,  86.67309166,  85.

In [ ]:
%time 
noise_var = 0.1
n_units = 128
n_hidden = 1
activation_fn = "tanh"
num_iters = 300  # Number of iteterations of Wasserstein optimization
lr = 0.05        # The learning rate
n_samples = 128  # The mini-batch size
out_dir = "./exp/gdpr/optim_gaussian"

X_pre_n, y_pre_n, y_mean, y_std = normalize_data(X_pre_ericcson, y_pre_ericcson)
x_min, x_max = get_input_range(X_pre_n, X_pre_n)
epsilon = 1e-6
x_min = np.minimum(x_min, x_max - epsilon)
input_dim, output_dim = int(X_pre_ericcson.shape[-1]), 1
    
# Initialize the measurement set generator
rand_generator = MeasureSetGenerator(X_pre_n, x_min, x_max, 0.7)

# Initialize the mean and covariance function of the target hierarchical GP prior
mean = mean_functions.Zero()
    
lengthscale = math.sqrt(2. * input_dim)
variance = 1.
kernel = kernels.RBF(input_dim=input_dim,
                     lengthscales=torch.tensor([lengthscale], dtype=torch.double),
                     variance=torch.tensor([variance], dtype=torch.double), ARD=True)

# Place hyper-priors on lengthscales and variances
kernel.lengthscales.prior = priors.LogNormal(
    torch.ones([input_dim]) * math.log(lengthscale),
    torch.ones([input_dim]) * 1.)
kernel.variance.prior = priors.LogNormal(
    torch.ones([1]) * 0.1,
    torch.ones([1]) * 1.)
        
# Initialize the GP model
gp = GPR(X=torch.from_numpy(X_pre_n), Y=torch.from_numpy(y_pre_n).reshape([-1, 1]),
             kern=kernel, mean_function=mean)
gp.likelihood.variance.set(noise_var)
    
# Initialize tunable MLP prior
hidden_dims = [n_units] * n_hidden
mlp_reparam = GaussianMLPReparameterization(input_dim, output_dim,
    hidden_dims, activation_fn, scaled_variance=True)
    
mapper = MapperWasserstein(gp, mlp_reparam, rand_generator, out_dir=out_dir,
                               output_dim=output_dim, n_data=100,
                               wasserstein_steps=(0, 300), ###more than 200
                               wasserstein_lr=0.02,
                               logger=None, wasserstein_thres=0.02,
                               n_gpu=1, gpu_gp=True) ##Change GPU if you don't have CUDA; same thing for the post training
    
w_hist = mapper.optimize(num_iters=num_iters, n_samples=n_samples,
                             lr=lr, print_every=10, save_ckpt_every=10, debug=True)

print("----" * 20)

In [ ]:
for name, param in mlp_reparam.named_parameters():
    print(f"parameter name: {name}, parameter shape: {param}")

In [ ]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

pre_weight_prior = [maintain_positivity(4.7054), maintain_positivity(1.1502)]
pre_bias_prior = [maintain_positivity(3.7491), maintain_positivity(-1.8061)]

In [ ]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

pre_weight_prior = [maintain_positivity(4.7054), maintain_positivity(1.1502)]
pre_bias_prior = [maintain_positivity(3.7491), maintain_positivity(-1.8061)]

In [ ]:
%time
num_epochs = 20001 # number of training epoches, 10000 now for quick test: 25000
progress_bar = trange(num_epochs) # show progress bar (only for visualization purpose)

X_pre_n_tensor = torch.tensor(X_pre_n, dtype=torch.float)
y_pre_n_tensor = torch.tensor(y_pre_n, dtype=torch.float)

for epoch in progress_bar:
    loss = svi.step(X_pre_n_tensor, y_pre_n_tensor)
    progress_bar.set_postfix(loss=f"{loss / X_pre_n.shape[0]:.3f}")
    if epoch % 1000 == 0:
        print("[iteration %04d] loss: %.3f" % (epoch + 1, loss / X_pre_n.shape[0])) #14.178

In [ ]:
predictive = Predictive(model_VI, guide=guide, num_samples=1000)

preds = predictive(X_pre_n_tensor)

plot_predictions(preds, y_pre_n_tensor)

In [ ]:
#RMSE
pred_samples = preds["obs"]
pred_mean = pred_samples.mean(dim=0)
# Calculate RMSE
y_true = y_pre_n_tensor
rmse = torch.sqrt(torch.mean((pred_mean - y_true) ** 2))
print(rmse)

In [ ]:
plot_uncertainty(preds, y_pre_n_tensor)

#### Posterior BNN: Ericcsson

In [11]:
df_post_ericcson = pd.read_csv("postgdprMay2018_Ericcson.csv")
df_post_ericcson.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,target
0,261755983.0,63.8,10.0,1.0,1.721164,261755983.0,63.5,10.0,1.0,1.721164,...,62.4,10.0,1.0,1.721164,261755983.0,62.0,10.0,1.0,1.721164,78.598425
1,261755983.0,63.5,10.0,1.0,1.721164,261755983.0,62.7,10.0,1.0,1.721164,...,62.0,10.0,1.0,1.721164,261755983.0,62.3,10.0,1.0,1.721164,81.958706
2,261755983.0,62.7,10.0,1.0,1.721164,261755983.0,62.4,10.0,1.0,1.721164,...,62.3,10.0,1.0,1.721164,261755983.0,63.6,10.0,1.0,1.721164,82.863704
3,261755983.0,62.4,10.0,1.0,1.721164,261755983.0,62.0,10.0,1.0,1.721164,...,63.6,10.0,1.0,1.721164,261755983.0,65.7,10.0,1.0,1.721164,82.616020
4,261755983.0,62.0,10.0,1.0,1.721164,261755983.0,62.3,10.0,1.0,1.721164,...,65.7,10.0,1.0,1.721164,261755983.0,67.1,10.0,1.0,1.721164,83.207702


In [12]:
#Extract Values
X_post_ericcson = df_post_ericcson.iloc[:, :-1].values
y_post_ericcson = df_post_ericcson["target"].values

# Create a mask for rows without NaNs in either X or y
valid_mask = ~np.isnan(X_post_ericcson).any(axis=1) & ~np.isnan(y_post_ericcson)

# Apply mask to both X and y
X_post_ericcson = X_post_ericcson[valid_mask]
y_post_ericcson = y_post_ericcson[valid_mask]
X_post_ericcson, X_post_ericcson.shape, type(X_post_ericcson)

(array([[2.61755983e+08, 6.38000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.72116375e+00],
        [2.61755983e+08, 6.35000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.72116375e+00],
        [2.61755983e+08, 6.27000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.72116375e+00],
        ...,
        [2.61755983e+08, 8.51000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.74115404e+00],
        [2.61755983e+08, 8.55000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.74115404e+00],
        [2.61755983e+08, 8.70000000e+01, 1.00000000e+01, ...,
         1.00000000e+01, 1.00000000e+00, 1.74115404e+00]], shape=(389, 25)),
 (389, 25),
 numpy.ndarray)

In [13]:
y_post_ericcson, y_post_ericcson.shape, type(y_post_ericcson)

(array([ 78.59842488,  81.95870643,  82.86370442,  82.61602043,
         83.20770237,  86.1705548 ,  86.93477066,  84.77553639,
         84.53745071,  83.86188259,  84.30710554,  83.30927248,
         85.5364581 ,  84.67808543,  82.75911643,  81.61228306,
         81.90226657,  83.16005061,  82.89991043,  83.55026543,
         84.09792478,  85.06813587,  85.42516464,  83.94829104,
         84.12592007,  90.48415245,  91.15079534,  89.45393926,
         89.52157927,  89.23152271,  89.67287798,  87.17776196,
         86.03570498,  85.45388775,  84.1862738 ,  85.89494583,
         87.5031066 ,  86.10491824,  86.06956188,  86.84555184,
         87.68516836,  87.83171246,  87.63198959,  87.93022772,
         88.19735189,  88.46882608,  89.45382941,  89.15991533,
         90.50718405,  91.90097156,  92.13930397,  94.8089543 ,
         97.4819094 ,  97.65988795,  98.25783065,  98.12220925,
         96.22196082,  96.51852965,  97.86566536,  98.00929751,
         96.68044983, 100.5472203 ,  99.

In [ ]:
noise_var = 0.1
n_units = 128
n_hidden = 1
activation_fn = "tanh"
num_iters = 300  # Number of iteterations of Wasserstein optimization
lr = 0.05        # The learning rate
n_samples = 128  # The mini-batch size
out_dir = "./exp/gdpr/optim_gaussian"

X_post_n, y_post_n, y_mean, y_std = normalize_data(X_post_ericcson, y_post_ericcson)
x_min, x_max = get_input_range(X_post_n, X_post_n)
epsilon = 1e-6
x_min = np.minimum(x_min, x_max - epsilon)
input_dim, output_dim = int(X_post_ericcson.shape[-1]), 1
    
# Initialize the measurement set generator
rand_generator = MeasureSetGenerator(X_post_n, x_min, x_max, 0.7)
    
# Initialize the mean and covariance function of the target hierarchical GP prior
mean = mean_functions.Zero()
    
lengthscale = math.sqrt(2. * input_dim)
variance = 1.
kernel = kernels.RBF(input_dim=input_dim,
                     lengthscales=torch.tensor([lengthscale], dtype=torch.double),
                     variance=torch.tensor([variance], dtype=torch.double), ARD=True)

# Place hyper-priors on lengthscales and variances
kernel.lengthscales.prior = priors.LogNormal(
    torch.ones([input_dim]) * math.log(lengthscale),
    torch.ones([input_dim]) * 1.)
kernel.variance.prior = priors.LogNormal(
    torch.ones([1]) * 0.1,
    torch.ones([1]) * 1.)
        
# Initialize the GP model
gp = GPR(X=torch.from_numpy(X_post_n), Y=torch.from_numpy(y_post_n).reshape([-1, 1]),
             kern=kernel, mean_function=mean)
gp.likelihood.variance.set(noise_var)
    
# Initialize tunable MLP prior
hidden_dims = [n_units] * n_hidden
mlp_reparam = GaussianMLPReparameterization(input_dim, output_dim,
    hidden_dims, activation_fn, scaled_variance=True)
    
mapper = MapperWasserstein(gp, mlp_reparam, rand_generator, out_dir=out_dir,
                               output_dim=output_dim, n_data=100,
                               wasserstein_steps=(0, 300), ##Should be more than 200
                               wasserstein_lr=0.02,
                               logger=None, wasserstein_thres=0.1,
                               n_gpu=1, gpu_gp=True) ##Change GPU if you don't have CUDA; same thing for the PRE training
    
w_hist = mapper.optimize(num_iters=num_iters, n_samples=n_samples,
                             lr=lr, print_every=10, save_ckpt_every=10, debug=True)

print("----" * 20)

In [ ]:
for name, param in mlp_reparam.named_parameters():
    print(f"parameter name: {name}, parameter shape: {param}")

In [ ]:
def maintain_positivity(x):
    '''
    maintain the positivity of weight and bias standard derivations
    '''
    return np.log(1 + np.exp(x))

post_weight_prior = [maintain_positivity(2.8332), maintain_positivity(1.0969)]
post_bias_prior = [maintain_positivity(2.0003), maintain_positivity(-0.1846)]

In [ ]:
# clear parameters to ensure every training start from scratch
pyro.clear_param_store()

# set up BNN
model_VI = BNN(post_weight_prior, post_bias_prior, in_dim=25, out_dim=1, hid_dim=128, n_hid_layers=1)

#mean_field_guide = AutoDiagonalNormal(model_VI) # mean field variational inference
guide = AutoMultivariateNormal(model_VI) # use multivariate normal with full covariance to approxiamte posterior

# apply SGD to maximizing the ELBO
optimizer = pyro.optim.Adam({"lr": 0.001})
svi = SVI(model_VI, guide, optimizer, loss=Trace_ELBO())

# # clear parameters to avoid influencing others
pyro.clear_param_store()

In [ ]:
num_epochs = 20001 # number of training epoches, 10000 now for quick test
progress_bar = trange(num_epochs) # show progress bar (only for visualization purpose)

X_post_n_tensor = torch.tensor(X_post_n, dtype=torch.float)
y_post_n_tensor = torch.tensor(y_post_n, dtype=torch.float)

for epoch in progress_bar:
    loss = svi.step(X_post_n_tensor, y_post_n_tensor)
    progress_bar.set_postfix(loss=f"{loss / X_post_n.shape[0]:.3f}")
    if epoch % 1000 == 0:
        print("[iteration %04d] loss: %.3f" % (epoch + 1, loss / X_post_n.shape[0])) ##10.158

In [ ]:
predictive = Predictive(model_VI, guide=guide, num_samples=1000)
preds = predictive(X_post_n_tensor)
plot_predictions(preds, y_post_n_tensor)

In [ ]:
#RMSE
pred_samples = preds["obs"]
pred_mean = pred_samples.mean(dim=0)
# Calculate RMSE
y_true = y_post_n_tensor
rmse = torch.sqrt(torch.mean((pred_mean - y_true) ** 2))
print(rmse)

In [ ]:
plot_uncertainty(preds, y_post_n_tensor)